In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
DEV_PATH = "../data/raw/development.csv"
C_VALUE = 1.5

df = pd.read_csv(DEV_PATH)

df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("").astype(str)
df["timestamp"] = df["timestamp"].fillna("").astype(str)

df["text"] = (df["title"] + " " + df["article"]).str.lower()

df["n_tokens"]    = df["article"].str.split().str.len()
df["title_len"]   = df["title"].str.len()
df["article_len"] = df["article"].str.len()
df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)

NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]
df[NUM_COLS] = df[NUM_COLS].replace([np.inf, -np.inf], 0).fillna(0)

df["ts_missing"] = (df["timestamp"] == "0000-00-00 00:00:00").astype(int)
def make_baseline():
    pre = ColumnTransformer(
        transformers=[
            ("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
            ("w_tfidf", TfidfVectorizer(
                analyzer="word",
                ngram_range=(1, 2),
                min_df=2,
                max_df=0.9,
                sublinear_tf=True,
                max_features=250_000
            ), "text"),
            ("c_tfidf", TfidfVectorizer(
                analyzer="char_wb",
                ngram_range=(3, 5),
                min_df=2,
                max_df=0.9,
                sublinear_tf=True,
                max_features=300_000
            ), "text"),
            ("num", StandardScaler(), NUM_COLS),
        ],
        remainder="drop",
        n_jobs=-1
    )

    clf = LogisticRegression(
        C=C_VALUE,
        class_weight="balanced",
        max_iter=2000,
        n_jobs=-1
    )

    return Pipeline([("pre", pre), ("clf", clf)])
df_miss = df[df["ts_missing"] == 1].copy()

X = df_miss
y = df_miss["label"]

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, val_idx = next(skf.split(X, y))

X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

model = make_baseline()
model.fit(X_tr, y_tr)

y_pred = model.predict(X_va)

print("=== BASELINE | TIMESTAMP MISSING ===")
print(classification_report(y_va, y_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_va, y_pred))
df_nomiss = df[df["ts_missing"] == 0].copy()

X = df_nomiss
y = df_nomiss["label"]

train_idx, val_idx = next(skf.split(X, y))

X_tr, X_va = X.iloc[train_idx], X.iloc[val_idx]
y_tr, y_va = y.iloc[train_idx], y.iloc[val_idx]

model = make_baseline()
model.fit(X_tr, y_tr)

y_pred = model.predict(X_va)

print("=== BASELINE | NO TIMESTAMP MISSING ===")
print(classification_report(y_va, y_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_va, y_pred))
